# Integer-Valued Sample Allocation Strategies for Multifidelity Monte Carlo

This notebook compares several integer-valued MFMC sample allocation strategies 
under a fixed computational budget.

Methods examined:

1. Real-valued optimal allocation (Peherstorfer closed form)
2. Naive floor allocation with modified ceil–floor allocation
3. Iterative allocation with duplicate removal
4. Adaptive variance-minimizing allocation

Five benchmark examples are drawn from published MFMC literature.

---------------------------------
Latest update: Mar-02, 2026,
Author: Jiaxing Liang

## Import packages

In [4]:
import numpy as np
from itertools import combinations
from scipy.optimize import minimize

## Model selection

- Exhausted approach

    The algorithm:
    
    1. Sorts models by decreasing absolute correlation: $|\rho_{1,1}|>\ldots>|\rho_{1,K}|$
    2. Removes duplicate correlation magnitudes.
    3. Enumerates all subsets containing the high-fidelity model.
    4. Enforces the MFMC ordering condition: $\frac{ \rho_{1,k}^2 - \rho_{1,k+1}^2 }{C_k} > \frac{ \rho_{1,k-1}^2 - \rho_{1,k}^2 }{C_{k-1}}, \quad  k=2,\ldots,K, $
    5. Selects the subset minimizing the cost-efficiency:
    $
    \xi = \frac{1}{C_1} 
    \left( \sum_{k \in \mathcal{S}} 
    \sqrt{C_k \left(\rho_k^2 - \rho_{k+1}^2\right)} \right)^2.
    $
    

In [5]:
def mfmc_model_selection_exhaustive(rho, C):
    """
    Exhaustive MFMC model selection.

    Parameters
    ----------
    rho : array_like
        Correlation coefficients.
    C : array_like
        Cost per sample.

    Returns
    -------
    ind_for_model : ndarray
        Indices of selected models (original indexing).
    xi_star : float
        Minimum cost-efficiency value.
    """

    rho = np.abs(np.array(rho))
    C = np.array(C)

    # -------------------------------------------------------------
    # Step 1: Sort by decreasing |rho|
    # -------------------------------------------------------------
    order = np.argsort(-rho)
    rho = rho[order]
    C = C[order]

    # -------------------------------------------------------------
    # Step 2: Remove duplicate |rho|
    # -------------------------------------------------------------
    unique_mask = np.ones(len(rho), dtype=bool)
    for i in range(len(rho) - 1):
        if rho[i] == rho[i + 1]:
            unique_mask[i + 1] = False

    rho = rho[unique_mask]
    C = C[unique_mask]
    order = order[unique_mask]

    K = len(rho)
    indices = np.arange(K)

    xi_star = np.inf
    best_subset = None

    # -------------------------------------------------------------
    # Step 3: Enumerate all subsets containing HF model (index 0)
    # -------------------------------------------------------------
    for r in range(1, K + 1):
        for subset in combinations(indices, r):

            if subset[0] != 0:
                continue  # must include HF model

            subset = np.array(subset)

            # -----------------------------------------------------
            # Case 1: Single high-fidelity model
            # -----------------------------------------------------
            if len(subset) == 1:
                xi = 1

            else:
                k = subset[:-1]
                k_p1 = subset[1:]

                # rho_{k+2}, last one maps to zero
                rho_k_p2 = np.append(rho[subset[2:]], 0)

                # -------------------------------------------------
                # MFMC ordering condition
                # -------------------------------------------------
                lhs = C[k] / C[k_p1]
                rhs = (rho[k]**2 - rho[k_p1]**2) / (rho[k_p1]**2 - rho_k_p2**2)

                if np.any(lhs <= rhs):
                    continue

                # -------------------------------------------------
                # Compute cost-efficiency xi
                # -------------------------------------------------
                rho_next = np.append(rho[subset[1:]], 0)
                xi = (np.sum(np.sqrt(C[subset] * (rho[subset]**2 - rho_next**2))) ** 2) / C[0]

            # -----------------------------------------------------
            # Update global minimum
            # -----------------------------------------------------
            if xi <= xi_star:
                xi_star = xi
                best_subset = subset

    # Map indices back to original indexing
    ind_for_model = order[best_subset]

    return ind_for_model, xi_star

## Real-Valued Allocation

In [6]:
def real_valued_allocation(rho, C, p):
    rho_p1 = np.append(rho[1:], 0)
    delta = rho**2 - rho_p1**2

    N_star = np.sqrt(delta / C) * p / np.sum(np.sqrt(C * delta))
    variance = np.sum(delta / N_star)
    cost = np.sum(C * N_star)

    return N_star, variance, cost, delta

## Integer Strategies

- Naive Floor: Peherstorfer
  
  Take floor after obtaining the real valued solution 

In [7]:
def naive_floor(C, delta, N_star):
    N_floor = np.floor(N_star)
    cost = np.sum(C * N_floor)
    try:
        with np.errstate(divide='raise', invalid='raise'): # Raise error if division by zero occurs
            variance = np.sum(delta / N_floor)
    except FloatingPointError:
        variance = np.inf
    return N_floor.astype(int), variance, cost

- Floor with modification: Gruber

  At least one sample size each entry

In [8]:
def modified_floor_allocation(C, delta, p, N_floor):
    """
    Modified floor-based integer allocation (Gruber-style correction).

    If any leading entries of the floor solution are < 1,
    they are fixed to 1 and the remaining allocation is recomputed
    using the analytic MFMC closed-form solution.

    Parameters
    ----------
    C : ndarray
        Cost vector.
    delta : ndarray
        Variance increments.
    p : float
        Total budget.
    N_floor : ndarray
        Initial floor allocation.

    Returns
    -------
    N_floor_modified : ndarray
        Corrected integer allocation.
    f_value : float
        Objective value sum(delta / N).
    total_cost : float
        Total cost sum(C * N).
    """

    K = len(C)
    N_floor_modified = N_floor.copy()
    j = 0  

    while j < K and N_floor_modified[j] < 1:

        # Fix current component to 1
        N_floor_modified[j] = 1

        # Remaining budget after fixing first j+1 entries
        used_budget = np.dot(C[:j+1], N_floor_modified[:j+1])
        remaining_budget = p - used_budget

        if remaining_budget <= 0:
            break

        # Recompute remaining entries via closed-form solution
        if j + 1 < K:
            delta_rem = delta[j+1:]
            C_rem = C[j+1:]
            N_floor_modified[j+1:] = np.floor(np.sqrt(delta_rem / C_rem) * remaining_budget / np.sum(np.sqrt(C_rem * delta_rem)))

        j += 1

    # Objective and cost
    f_value = np.sum(delta / N_floor_modified)
    total_cost = np.dot(C, N_floor_modified)

    return N_floor_modified, f_value, total_cost

- Iterative Allocation: 

  Dynamic programing

  $$
  y_1
=
\sqrt{\frac{\Delta_1}{C_1}}\frac{b}{S_1},
\qquad
y_k
=
\sqrt{\frac{\Delta_k}{C_k}}
\frac{b-\sum_{j=1}^{k-1} C_j \lfloor y_j \rfloor}{S_k},
\quad k=2,\ldots,K,
  $$

  The procedure begins with the continuous allocation $y_1$, from which the integer sample size $\lfloor y_1 \rfloor$ is obtained. The associated cost $C_1 \lfloor y_1 \rfloor$ is deducted from the total budget, and the recursion proceeds to the next fidelity level using the remaining budget. In this manner, each integer allocation $\lfloor y_k \rfloor$ is constructed adaptively with respect to the updated residual budget.

In [9]:
def iterative_allocation(C, delta, p):
    K = len(C)
    N = np.zeros(K, dtype=int)

    N_star = np.sqrt(delta / C) * p / np.sum(np.sqrt(C * delta))
    N[0] = max(np.floor(N_star[0]).astype(int), 1)

    for k in range(1, K):
        remaining_budget = p - np.sum(C[:k] * N[:k])
        denom = np.sum(np.sqrt(C[k:] * delta[k:]))
        Nk = np.floor(np.sqrt(delta[k]/C[k]) * remaining_budget / denom)
        N[k] = max(Nk.astype(int), 1)

    variance = np.sum(delta / N)
    cost = np.sum(C * N)

    return N, variance, cost

- Iterative allocation with post-processing to delete equal:

  if delete_equal = False, this algorithm is iterative allocation.
  
  if delete_equal = True, this algorithm delete equal integer sample size whenever it detects any.

In [10]:
def iterative_allocation_delete_equal(rho, C, delta, p, delete_equal=True):
    """
    Iterative integer allocation with optional model deletion and restart mechanism.

    Parameters
    ----------
    rho : ndarray
        Correlation coefficients (ordered).
    C : ndarray
        Cost per model.
    delta : ndarray
        Variance increments.
    p   : float
        Total cost budget.
    delete_equal : bool, optional
        Whether to discard models when equality condition occurs.

    Returns
    -------
    rho : ndarray
        Possibly reduced correlation vector.
    C : ndarray
        Possibly reduced cost vector.
    N_iter : ndarray
        Final integer allocation.
    objective : float
        sum(delta / N_iter)
    total_cost : float
        sum(C * N_iter)
    """

    rho = np.asarray(rho, dtype=float).copy()
    C = np.asarray(C, dtype=float).copy()
    N_star = np.sqrt(delta / C) * p / np.sum(np.sqrt(C * delta))

    restart = True

    while restart:
        restart = False

        K = len(rho)

        # Compute delta = rho_k^2 - rho_{k+1}^2
        rho_p1 = np.append(rho[1:], 0.0)
        delta = rho**2 - rho_p1**2

        N_iter = np.zeros(K, dtype=int)

        # First entry
        N_iter[0] = max(int(np.floor(N_star[0])), 1)

        for k in range(1, K):

            used_budget = np.dot(C[:k], N_iter[:k])
            remaining_budget = p - used_budget

            if remaining_budget <= 0:
                break

            # Closed-form MFMC allocation for stage k
            denom = np.sum(np.sqrt(C[k:] * delta[k:]))
            if denom <= 0:
                N_iter[k] = 1
            else:
                val = (
                    np.sqrt(delta[k] / C[k])
                    * remaining_budget
                    / denom
                )
                N_iter[k] = max(int(np.floor(val)), 1)

            # ----------------------------
            # Model deletion logic
            # ----------------------------
            if delete_equal and k < K - 1:

                if N_iter[k] == N_iter[k - 1]:

                    delta_km1 = rho[k - 1]**2 - rho[k + 1]**2
                    delta_k = rho[k + 1]**2 - 0.0  # next-next term

                    left_ratio = delta_km1 / C[k - 1]
                    right_ratio = delta_k / C[k + 1]

                    if left_ratio < right_ratio:
                        # Delete current model k
                        rho = np.delete(rho, k)
                        C = np.delete(C, k)

                        restart = True
                        # print(f"Delete model: {k+1}")
                        break

        # Restart loop if deletion occurred
        if restart:
            continue

    # Final recomputation after convergence
    rho_p1 = np.append(rho[1:], 0.0)
    delta = rho**2 - rho_p1**2

    variance = np.sum(delta / N_iter)
    total_cost = np.dot(C, N_iter)

    return rho, C, N_iter, variance, total_cost

- Adaptive Sequential Optimization for Integer MFMC Allocation:

We solve the integer-constrained MFMC allocation problem via a 
stage-wise adaptive strategy.

At each stage j:

1. Solve the relaxed continuous subproblem.
2. Decide between floor/ceil for m_j.
3. Re-optimize future stages.
4. Enforce monotonicity: m_j ≥ m_{j-1}.
5. Respect total budget constraint.

This produces a globally consistent integer allocation
without solving a full integer program.

$$
\begin{align}
    \min \quad & \sum_{k=1}^{j-1}\frac{\Delta_k}{\overline{m}_k}
    + \frac{\Delta_j}{x_j}
    + \ldots + \frac{\Delta_K}{x_K},\\
    % \text{s.t.} \quad & \sum_{k=j}^K C_k m_k=C_jm_j+\sum_{k=j+1}^K\sqrt{C_k\Delta_k}\frac{b_k}{\sum_{i=k}^K\sqrt{C_i\Delta_i}} \le b_j=b - \sum_{k=1}^{j-1} C_k\overline{m}_k, \\
    \text{s.t.} \quad &\sum_{k=j}^K C_k x_k \le b - \sum_{k=1}^{j-1} C_k\overline{m}_k, \\
    & x_j \ge \overline{m}_{j-1},\qquad x_k \ge x_{k-1}, \quad k=j+1,\ldots,K, \label{eq:Adaptive_Optimization_obj_c}\\
    & x_j,\ldots,x_K \in \mathbb{R}.
\end{align}
$$

-Adaptive strategy: Continuous Subproblem

In [11]:
def solve_subproblem(j, C, Delta, remaining_budget, m_fixed):
    """
    Solve relaxed continuous subproblem for stages j,...,K.

    Parameters
    ----------
    j : int
        Current stage index (0-based).
    C : ndarray
        Cost vector.
    Delta : ndarray
        Variance increments.
    remaining_budget : float
        Budget available from stage j onward.
    m_fixed : ndarray
        Integer values fixed for stages < j.

    Returns
    -------
    m_opt : ndarray
        Optimal continuous allocation for j,...,K.
    fval : float
        Objective value including fixed contribution.
    """

    K = len(C)
    n = K - j

    # Constant contribution from previous fixed integers
    const_term = np.sum(Delta[:j] / m_fixed[:j]) if j > 0 else 0.0

    Delta_j = Delta[j:]
    C_j = C[j:]

    def objective(x):
        return const_term + np.sum(Delta_j / x)

    # Lower bounds (monotonicity constraint)
    lb = np.ones(n)
    if j > 0:
        lb[0] = m_fixed[j-1]

    bounds = [(lb[i], None) for i in range(n)]

    # Budget constraint
    cons = {
        'type': 'ineq',
        'fun': lambda x: remaining_budget - np.dot(C_j, x)
    }

    # Initialization via analytic relaxed structure
    x0 = np.sqrt(Delta_j / C_j)*remaining_budget / np.sum(np.sqrt(C_j * Delta_j))
    x0 = np.maximum(x0, lb)

    result = minimize(
        objective,
        x0,
        method='SLSQP',
        bounds=bounds,
        constraints=[cons],
        options={'ftol': 1e-10, 'disp': False}
    )

    return result.x, result.fun

## 

In [12]:
def decide_integer(j, m_real_j, C, Delta, remaining_budget, m_int):
    """
    Decide integer value for stage j using floor/ceil evaluation.

    Parameters
    ----------
    j : int
        Current stage index (0-based).
    m_real_j : float
        Continuous solution at stage j.
    C : ndarray
        Cost vector.
    Delta : ndarray
        Variance increments.
    remaining_budget : float
        Available budget from stage j onward.
    m_int : ndarray
        Current partially fixed integer allocation.

    Returns
    -------
    best_choice : int
        Selected integer allocation for stage j.
    """

    # Candidate integers
    candidates = np.unique([np.floor(m_real_j), np.ceil(m_real_j)]).astype(int)

    best_var = np.inf
    best_choice = None

    for cand in candidates:

        # ---- Feasibility checks ----
        if cand < 1:
            continue

        # Monotonicity constraint
        if j > 0 and cand < m_int[j - 1]:
            continue

        # Budget feasibility
        if C[j] * cand > remaining_budget:
            continue

        temp_m = m_int.copy()
        temp_m[j] = cand

        # ---- Evaluate downstream objective ----
        if j < len(C) - 1:
            _, fval = solve_subproblem(j + 1, C, Delta, remaining_budget - C[j] * cand, temp_m)
            total_var = fval
        else:
            total_var = np.sum(Delta / temp_m)

        if total_var < best_var:
            best_var = total_var
            best_choice = cand

    # Fallback safeguard (rare but numerically possible)
    if best_choice is None:
        best_choice = max(1, np.floor(m_real_j).astype(int))

    return best_choice

In [13]:
def adaptive_allocation(C, Delta, b):
    """
    Adaptive sequential integer allocation for MFMC.

    Returns
    -------
    m_real : ndarray
        Continuous solution per stage.
    m_int : ndarray
        Final integer allocation.
    fval_vec : ndarray
        Objective value at each stage.
    """

    K = len(C)

    m_int = np.ones(K, dtype=int)
    m_real = np.zeros(K)
    fval_vec = np.zeros(K)

    used_budget = 0

    for j in range(K):

        remaining_budget = b - used_budget
        if remaining_budget <= 0:
            break

        # Continuous relaxation
        m_opt_real, fval = solve_subproblem(j, C, Delta, remaining_budget, m_int)

        m_real[j] = m_opt_real[0]
        fval_vec[j] = fval

        # Integer decision
        m_j_int = decide_integer(j, m_real[j], C, Delta, remaining_budget, m_int)

        m_int[j] = m_j_int
        used_budget += C[j] * m_j_int

    return m_int, fval_vec[-1], used_budget

In [14]:
def adaptive_allocation_delete_equal(C, rho, delta, b, delete_equal=True):
    """
    Adaptive sequential integer allocation with optional
    delete-equal (Liang-style) model pruning.

    Parameters
    ----------
    C : ndarray
        Cost per model.
    rho : ndarray
        Correlation coefficients (ordered).
    delta : ndarray
        Variance increments (rho_k^2 - rho_{k+1}^2).
    b : float
        Total budget.
    delete_equal : bool, optional
        Whether to discard models when equal allocation occurs.

    Returns
    -------
    C : ndarray
        Possibly reduced cost vector.
    rho : ndarray
        Possibly reduced correlation vector.
    m_int : ndarray
        Final integer allocation.
    objective : float
        Final variance value.
    total_cost : float
        Total cost used.
    """

    C = np.asarray(C, dtype=float).copy()
    rho = np.asarray(rho, dtype=float).copy()

    restart = True

    while restart:
        restart = False

        K = len(C)

        # Recompute delta from rho
        rho_p1 = np.append(rho[1:], 0.0)
        delta = rho**2 - rho_p1**2

        m_int = np.ones(K, dtype=int)
        m_real = np.zeros(K)

        used_budget = 0

        for j in range(K):

            remaining_budget = b - used_budget
            if remaining_budget <= 0:
                break

            # Continuous relaxation
            m_opt_real, fval = solve_subproblem(
                j, C, delta, remaining_budget, m_int
            )

            m_real[j] = m_opt_real[0]

            # Integer decision
            m_j_int = decide_integer(
                j, m_real[j], C, delta,
                remaining_budget, m_int
            )

            m_int[j] = m_j_int
            used_budget += C[j] * m_j_int

            # ----------------------------------
            # Delete-equal logic
            # ----------------------------------
            if delete_equal and j > 0 and j < K - 1:

                if m_int[j] == m_int[j - 1]:

                    delta_km1 = rho[j - 1]**2 - rho[j + 1]**2
                    delta_k = rho[j + 1]**2 - 0.0

                    left_ratio = delta_km1 / C[j - 1]
                    right_ratio = delta_k / C[j + 1]

                    if left_ratio < right_ratio:
                        # Remove model j
                        C = np.delete(C, j)
                        rho = np.delete(rho, j)

                        restart = True
                        break

        # Restart outer loop if model deleted
        if restart:
            continue

    # Final objective evaluation
    rho_p1 = np.append(rho[1:], 0.0)
    delta = rho**2 - rho_p1**2

    variance = np.sum(delta / m_int)
    total_cost = np.dot(C, m_int)

    return C, rho, m_int, variance, total_cost

## Test examples

the total computational budget must satisfy the following condition to guarantee that each fidelity level receives at least one sample

$$
    b \ge \sum_{k=1}^K C_k,
$$


In [30]:
def load_example(example_name):
    """
    Load benchmark MFMC examples from the literature.

    Returns
    -------
    rho : ndarray
        Correlation coefficients (selected models).
    C : ndarray
        Cost per sample (selected models).
    budget : float
        Total available budget.
    """

    if example_name == "Example_1":
        # Jiaxing et al.
        # Multilevel Monte Carlo methods for the Grad-Shafranov free boundary problem
        # DOI: 10.1016/j.cpc.2024.109099
        # Data after model selection: plasma model
        rho = np.array([1, 9.9977e-01, 9.9925e-01, 9.9728e-01, 9.8390e-01])
        C   = np.array([7.30e+01, 7.0318e-03, 1.4018e-03,
                        5.0613e-04, 2.6803e-04])
        sigma1 = 1.0840e-02 #standard deviation sigma1
        budget = 248.0

    elif example_name == "Example_2":
        # Peherstorfer et al. (2016)
        # Optimal model management for multifidelity Monte Carlo estimation
        # DOI: 10.1137/15M1046472
        # Data after model selection:tubular reactor problem
        rho = np.array([1, 9.999882e-01, 9.999743e-01, 9.958253e-01])
        C   = np.array([44.395, 6.8409e-01, 2.9937e-01, 1.9908e-04])
        # no sigma1 mentioned in the paper
        budget = 46.0

    elif example_name == "Example_3":
        # Konrad, Julia et al.
        # Data-driven low-fidelity models for multi-fidelity Monte Carlo
        # sampling in plasma micro-turbulence analysis
        # DOI: 10.1016/j.jcp.2021.110898
        # Data after model selection: models for modified Cyclone Base case
        rho = np.array([1, 0.9819, 0.9708])
        C   = np.array([240.5123, 0.0166, 0.0017])
        sigma1 = 0.0754
        # budget = 5e2, 1e3, 5e3, 1e4, 5e4 
        budget = 960.0

    elif example_name == "Example_4":
        # Qian, E.
        # Multifidelity Monte Carlo estimation of variance and sensitivity indices
        # DOI: 10.1137/17M1151006
        # Data after model selection:  Ishigami function
        rho = np.array([1, 0.9997, 0.9465])
        C   = np.array([1, 0.05, 0.001])
        sigma1 = 3.29
        # budget = 2e2 ~ 2e4 
        budget = 2000.0

    elif example_name == "Example_5":
        # Gorodetsky (2020)
        # Generalized approximate control variates framework
        
        rho = np.array([1, 0.99838, 0.99245, 0.96560, 0.70267])
        C   = np.array([1.000, 0.147, 0.026, 0.009, 0.002])
        budget = 51.0

    elif example_name == "Example_6":
        # Peherstorfer et al. (2016)
        # Optimal model management for multifidelity Monte Carlo estimation
        # DOI: 10.1137/15M1046472
        # Data after model selection: plate problem
        rho = np.array([1, 9.99999983e-01, 9.99999216e-01, 9.99954506e-01, 9.98971009e-01, 9.97555261e-01])
        C   = np.array([4.0894e-01, 4.9890e-03, 1.3264e-03, 2.9550e-04, 2.2260e-05, 1.5048e-06])
        sigma1 =  1.5511e-02
        # budget = 1~1e3
        budget = 46.0

    elif example_name == "Example_7":
        # Anthony Gruber et al. (2022)
        # A Multifidelity Monte Carlo Method for Realistic Computational Budgets
        # Data before model selection: Table 1
        rho = np.array([1.0000000, 0.99994645, 0.6980721, 0.92928154, 0.99863737])
        C   = np.array([100, 50, 20, 10, 5])
        sigma1 = 6.9520  # no sigma1 mentioned in the paper, computed from sample size [2, 17, 1059]
        # budget = 2*C[1]~64*C[1]
        budget = 2*100

    elif example_name == "Example_8":
        # Anthony Gruber et al. (2022)
        # A Multifidelity Monte Carlo Method for Realistic Computational Budgets
        # Data before model selection: Table 2
        # rho = np.array([1.0000000, 0.99766585, 0.98343683, 0.99999507, 0.99999882])
        # C   = 1e-04*np.array([30.5625, 5.5174, 5.8633, 6.3854, 7.4522])
        rho = np.array([1.0000000, 0.99999882, 0.99999507, 0.99766585, 0.98343683])
        C   = 1e-04*np.array([30.5625, 7.4522, 6.3854, 5.5174, 5.8633])
        sigma1 = 7.0678 # no sigma1 mentioned in the paper, computed from sample size [1, 20, 325]
        # budget = 2*C[1]~64*C[1]
        budget = 2*100

    return rho, C, budget

In [33]:
rho, C, budget = load_example("Example_8")
selected, xi_star = mfmc_model_selection_exhaustive(rho, C)
print(selected)

rho = rho[selected]
C   = C[selected]

N_star, f_real, cost_real, delta = real_valued_allocation(rho, C, budget)
N_floor, f_floor, cost_floor = naive_floor(C, delta, N_star)
N_floor_modified, f_floor_modified, cost_floor_modified = modified_floor_allocation(C, delta, budget, N_floor)

N_iter, f_iter, cost_iter = iterative_allocation(C, delta, budget)
_,_,N_iter_delete_equal, f_iter_delete_equal, cost_iter_delete_equal = iterative_allocation_delete_equal(rho, C, delta, budget,delete_equal=True)

N_adapt, f_adapt, cost_adapt = adaptive_allocation(C, delta, budget)
_,_,N_adapt_delete_equal, f_adapt_delete_equal, cost_adapt_delete_equal= adaptive_allocation_delete_equal(C, rho, delta, budget, delete_equal=True)

[0 4 3 1]


## Results Table

In [28]:
print("===== Real-valued solution =====")
formatted_N = ", ".join(f"{x:.5e}" for x in N_star)
print(f"N = [{formatted_N}]")
print("Variance = {:.5e}".format(f_real))
print("Cost = {:.5e}".format(cost_real))

print("\n===== Naive floor =====")
formatted_N = ", ".join(f"{x}" for x in N_floor)
print(f"N = [{formatted_N}]")
print("Variance = {:.5e}".format(f_floor))
print("Cost = {:.5e}".format(cost_floor))

print("\n===== Floor with modification =====")
formatted_N = ", ".join(f"{x}" for x in N_floor_modified)
print(f"N = [{formatted_N}]")
print("Variance = {:.5e}".format(f_floor_modified))
print("Cost = {:.5e}".format(cost_floor_modified))

print("\n===== Iterative =====")
formatted_N = ", ".join(f"{x}" for x in N_iter)
print(f"N = [{formatted_N}]")
print("Variance = {:.5e}".format(f_iter))
print("Cost = {:.5e}".format(cost_iter))

print("\n===== Iterative with delete equal=====")
formatted_N = ", ".join(f"{x}" for x in N_iter_delete_equal)
print(f"N = [{formatted_N}]")
print("Variance = {:.5e}".format(f_iter_delete_equal))
print("Cost = {:.5e}".format(cost_iter_delete_equal))

print("\n===== Adaptive =====")
formatted_N = ", ".join(f"{x}" for x in N_adapt)
print(f"N = [{formatted_N}]")
print("Variance = {:.5e}".format(f_adapt))
print("Cost = {:.5e}".format(cost_adapt))

print("\n===== Adaptive with delete equal=====")
formatted_N = ", ".join(f"{x}" for x in N_adapt_delete_equal)
print(f"N = [{formatted_N}]")
print("Variance = {:.5e}".format(f_adapt_delete_equal))
print("Cost = {:.5e}".format(cost_adapt_delete_equal))

===== Real-valued solution =====
N = [7.67089e-02, 5.36187e-01, 3.31040e+01]
Variance = 3.64012e-02
Cost = 2.00000e+02

===== Naive floor =====
N = [0, 0, 33]
Variance = inf
Cost = 1.65000e+02

===== Floor with modification =====
N = [1, 1, 10]
Variance = 1.02451e-01
Cost = 2.00000e+02

===== Iterative =====
N = [1, 1, 10]
Variance = 1.02451e-01
Cost = 2.00000e+02

===== Iterative with delete equal=====
N = [1, 20]
Variance = 5.25872e-02
Cost = 2.00000e+02

===== Adaptive =====
N = [1, 1, 10]
Variance = 1.02451e-01
Cost = 2.00000e+02

===== Adaptive with delete equal=====
N = [1, 20]
Variance = 5.25872e-02
Cost = 2.00000e+02
